<a href="https://colab.research.google.com/github/wellanx/ml-olympiad-red-task/blob/main/ml_olympiad_red_task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 0. Установка зависимостей




In [1]:
!pip -q install -U \
    "transformers==4.55.2" \
    "trl==0.21.0" \
    "peft==0.17.0" \
    "bitsandbytes==0.47.0" \
    "accelerate==1.10.0" \
    "datasets==3.6.0" \
    "scikit-learn==1.7.2" \
    "scipy" "safetensors" "sentencepiece"
print("install done")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 98.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.9/511.9 kB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.9/503.9 kB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.7/374.7 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [2]:
!unzip -q ml-olympiad-red-task-c1005bf0-8695-451a-9616-87c8687dce27

## 1. Конфиг и фиксация seed




In [3]:
import os, random, json, gc
import numpy as np
import torch
from transformers import set_seed

SEED = 42
def fix_seed(s=SEED):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    set_seed(s)
fix_seed()

DATA_DIR  = "ml-olympiad-red-task"
CKPT_DIR  = "checkpoints"
MODEL_ID   = "Qwen/Qwen3-4B-Instruct-2507"
MAX_LEN    = 1024
GEN_MAX_NEW= 256
DATA = os.path.join(DATA_DIR, "data")
METR = os.path.join(DATA_DIR, "metrics")

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

kid_adult = load_jsonl(os.path.join(DATA, "kid_adult.jsonl"))
good_bad  = load_jsonl(os.path.join(DATA, "good_bad.jsonl"))
test_style   = load_jsonl(os.path.join(DATA, "public_test_style.jsonl"))
test_quality = load_jsonl(os.path.join(DATA, "public_test_quality.jsonl"))
print(len(kid_adult), len(good_bad), len(test_style), len(test_quality))
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

1489 2226 50 50
device: Tesla T4


## 2. Токенайзер, загрузчик 4-bit модели и общие хелперы

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

def load_base(for_seqclass=False, num_labels=1):
    kw = dict(quantization_config=bnb_config, torch_dtype=torch.float16, device_map={"": 0})
    if for_seqclass:
        from transformers import AutoModelForSequenceClassification
        m = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, num_labels=num_labels, **kw)
        m.config.pad_token_id = tokenizer.pad_token_id
    else:
        m = AutoModelForCausalLM.from_pretrained(MODEL_ID, **kw)
    return m

def free(*objs):
    for o in objs:
        try: del o
        except Exception: pass
    gc.collect(); torch.cuda.empty_cache()

def prompt_str(user_text):
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": user_text}],
        tokenize=False, add_generation_prompt=True)

LORA_TARGETS = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

## 3. Измеритель стиля `P_simple`

In [5]:
import pickle
from scipy.sparse import hstack

with open(os.path.join(METR, "style_clf.pkl"), "rb") as f:
    _style = pickle.load(f)
_clf, _vecs = _style["clf"], _style["vecs"]

def p_simple(texts):
    X = hstack([v.transform(texts) for v in _vecs]).tocsr()
    return _clf.predict_proba(X)[:, 1]

_pk = p_simple([r["kid"]   for r in test_style]).mean()
_pa = p_simple([r["adult"] for r in test_style]).mean()
print(f"sanity  P_simple(kid)={_pk:.3f}  P_simple(adult)={_pa:.3f}  (ожидаем высокий/низкий)")

sanity  P_simple(kid)=0.975  P_simple(adult)=0.018  (ожидаем высокий/низкий)


## 4. Хелперы: greedy-генерация и length-normalized logprob

In [6]:
@torch.no_grad()
def generate_answers(model, prompts, max_new=GEN_MAX_NEW):
    fix_seed()
    model.eval()
    outs = []
    for p in prompts:
        ids = tokenizer(prompt_str(p), return_tensors="pt").input_ids.to(model.device)
        g = model.generate(
            ids, max_new_tokens=max_new,
            do_sample=False, num_beams=1,
            temperature=None, top_p=None, top_k=None,
            pad_token_id=tokenizer.eos_token_id,
        )
        outs.append(tokenizer.decode(g[0, ids.shape[1]:], skip_special_tokens=True).strip())
    return outs

@torch.no_grad()
def avg_token_logprob(model, user_text, response_text):
    model.eval()
    p_ids = tokenizer(prompt_str(user_text), return_tensors="pt").input_ids
    full  = prompt_str(user_text) + response_text
    f_ids = tokenizer(full, return_tensors="pt").input_ids.to(model.device)
    logits = model(f_ids).logits[:, :-1, :]
    logp   = torch.log_softmax(logits.float(), dim=-1)
    labels = f_ids[:, 1:]
    tok_lp = logp.gather(-1, labels.unsqueeze(-1)).squeeze(-1)[0]
    start  = p_ids.shape[1] - 1
    resp   = tok_lp[start:]
    return resp.mean().item()

def preference_accuracy(model, pairs):
    wins = 0
    for r in pairs:
        lc = avg_token_logprob(model, r["prompt"], r["chosen"])
        lr = avg_token_logprob(model, r["prompt"], r["rejected"])
        wins += int(lc > lr)
    return wins / len(pairs)